# CTG (Cross-Temporal Generalization) Stability Analysis Pipeline

This notebook is a thin wrapper around the `src.analysis.ctg_pipeline` module.

## Setup Instructions

1. Copy `config_default.yaml` to `config.yaml` and edit for your setup
2. If on Google Colab, mount your Google Drive
3. Run the cells below to execute the pipeline


In [ ]:
# Cell 1: Install dependencies and mount drive (if on Colab)

import sys
import os

# Check if running on Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab")
    
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Install required packages
    !pip install -q ONE-api scikit-learn joblib pyyaml numpy pandas matplotlib seaborn
    
except ImportError:
    IN_COLAB = False
    print("Running locally")

# Add penelope directory to path
penelope_dir = os.path.abspath('..')
if penelope_dir not in sys.path:
    sys.path.insert(0, penelope_dir)

print(f"Python path: {sys.path[:3]}")

In [ ]:
# Cell 2: Download IBL data

from src.analysis.ctg_pipeline import ConfigLoader, download_data

# Load configuration
config = ConfigLoader()

# Apply any environment overrides
config.apply_env_overrides()

print(f"Cache directory: {config.get_cache_dir()}")
print(f"Number of sessions: {len(config.get_eids())}")
print(f"Session EIDs: {config.get_eids()}")

# Download data
download_data(config)

In [ ]:
# Cell 3: Populate cache and validate data

from src.analysis.ctg_pipeline import populate_cache

populate_cache(config)

In [ ]:
# Cell 4: Run CTG stability analysis

from src.analysis.ctg_pipeline import run_analysis

run_analysis(config)

In [ ]:
# Cell 5: Visualize results (optional)

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load results for visualization
output_path = config.get_output_path()
eids = config.get_eids()

# Plot CTG matrix for first session
if eids:
    eid = eids[0]
    result_file = output_path / f'{eid}_ctg_results.npz'
    
    if result_file.exists():
        data = np.load(result_file)
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # CTG matrix
        im = axes[0].imshow(data['ctg_mean'], cmap='RdYlBu_r', aspect='auto')
        axes[0].set_title(f'CTG Matrix - {eid[:8]}')
        axes[0].set_xlabel('Test Time')
        axes[0].set_ylabel('Train Time')
        plt.colorbar(im, ax=axes[0], label='Accuracy')
        
        # Self-consistency
        axes[1].plot(data['self_consistency_mean'], label='Mean')
        axes[1].fill_between(
            range(len(data['self_consistency_mean'])),
            data['self_consistency_mean'] - data['self_consistency_std'],
            data['self_consistency_mean'] + data['self_consistency_std'],
            alpha=0.3
        )
        axes[1].set_title('Self-Consistency Over Time')
        axes[1].set_xlabel('Time Bin')
        axes[1].set_ylabel('Accuracy')
        axes[1].legend()
        
        plt.tight_layout()
        plt.show()
        
        print(f"Session: {eid}")
        print(f"Neurons: {data['n_neurons']}")
        print(f"Trials: {data['n_trials']}")
    else:
        print(f"Results not found: {result_file}")
else:
    print("No sessions configured")